# Depression Detection - Competition 2

## Structure
1. **Setup** - Imports and data loading
2. **EDA** - Exploratory data analysis
3. **Shared Functions** - Reusable helper functions
4. **Title Model** - TF-IDF on title with feature importance
5. **Body Model** - TF-IDF on body with feature importance
6. **Blend Title + Body** - Combine title and body predictions
7. **Char Model** - Character-level TF-IDF
8. **Blend Title + Body + Char** - 3-way model blend
9. **Handcrafted Features** - Dense features from lexicons
10. **Meta Model** - Combine all predictions
11. **Final Submission** - Best model selection

---

## 1. Setup - Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Paths
DATA_DIR = '.'

# Load data
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')

# Preprocess
for col in ['title', 'body']:
    train[col] = train[col].fillna('').astype(str)
    test[col] = test[col].fillna('').astype(str)

train['full_text'] = train['title'] + ' ' + train['body']
test['full_text'] = test['title'] + ' ' + test['body']

y = train['label'].astype(int).values

print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'Label distribution:\n{train["label"].value_counts()}')

---

## 2. EDA - Exploratory Data Analysis

In [ ]:
# Analyze text from depressed vs non-depressed class
dep = train[train['label'] == 1]['full_text']
non_dep = train[train['label'] == 0]['full_text']

# Top words in depressed class
cv = CountVectorizer(lowercase=True, stop_words='english', min_df=3, ngram_range=(1,1))
X_dep = cv.fit_transform(dep)

freq = np.asarray(X_dep.sum(axis=0)).ravel()
words = np.array(cv.get_feature_names_out())

top_dep = pd.DataFrame({'word': words, 'count_in_dep': freq}).sort_values('count_in_dep', ascending=False)

print('TOP WORDS IN DEPRESSED CLASS:')
print(top_dep.head(30).to_string(index=False))

# Log-odds ratio: words more specific to depression
cv2 = CountVectorizer(lowercase=True, stop_words='english', min_df=5, ngram_range=(1,1))
X_all = cv2.fit_transform(train['full_text'])
vocab = np.array(cv2.get_feature_names_out())

X1 = cv2.transform(dep)
X0 = cv2.transform(non_dep)

c1 = np.asarray(X1.sum(axis=0)).ravel() + 1  # smoothed
c0 = np.asarray(X0.sum(axis=0)).ravel() + 1

score = np.log(c1 / c1.sum()) - np.log(c0 / c0.sum())

top_signal = pd.DataFrame({
    'word': vocab,
    'log_odds_dep': score,
    'dep_count': c1 - 1,
    'nondep_count': c0 - 1
}).sort_values('log_odds_dep', ascending=False)

print('\nMOST DEPRESSION-SPECIFIC WORDS (log-odds):')
print(top_signal.head(30).to_string(index=False))

---

## 3. Shared Functions

In [ ]:
# CV setup
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)


def find_best_threshold(y_true, pred_proba, thr_min=0.10, thr_max=0.90, thr_step=0.01):
    """Find best threshold for F1 score."""
    best_thr = 0.5
    best_score = -1
    
    for thr in np.arange(thr_min, thr_max + 1e-9, thr_step):
        pred = (pred_proba >= thr).astype(int)
        score = f1_score(y_true, pred)
        if score > best_score:
            best_score = score
            best_thr = round(thr, 2)
    
    return best_thr, best_score


def train_oof_model(X_train_full, X_test_full, y, skf, feature_name, tfidf_params, clf_params):
    """Train model with OOF predictions."""
    oof_pred = np.zeros(len(X_train_full))
    test_pred_folds = np.zeros((len(X_test_full), skf.n_splits))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_full, y), 1):
        X_train = X_train_full.iloc[train_idx]
        X_valid = X_train_full.iloc[valid_idx]
        y_train = y[train_idx]
        y_valid = y[valid_idx]

        model = Pipeline([
            ('tfidf', TfidfVectorizer(**tfidf_params)),
            ('clf', LogisticRegression(**clf_params))
        ])

        model.fit(X_train, y_train)

        valid_pred = model.predict_proba(X_valid)[:, 1]
        test_pred = model.predict_proba(X_test_full)[:, 1]

        oof_pred[valid_idx] = valid_pred
        test_pred_folds[:, fold - 1] = test_pred

        fold_f1 = f1_score(y_valid, (valid_pred >= 0.5).astype(int))
        fold_scores.append(fold_f1)

        print(f'[{feature_name}] Fold {fold}: F1 = {fold_f1:.5f}')

    oof_f1_05 = f1_score(y, (oof_pred >= 0.5).astype(int))
    mean_cv_f1 = np.mean(fold_scores)

    best_thr, best_f1 = find_best_threshold(y, oof_pred)
    test_pred_mean = test_pred_folds.mean(axis=1)

    print(f'\n[{feature_name}] OOF F1 @ 0.50: {oof_f1_05:.5f}')
    print(f'[{feature_name}] Mean CV F1: {mean_cv_f1:.5f}')
    print(f'[{feature_name}] Best threshold: {best_thr:.2f}')
    print(f'[{feature_name}] Best OOF F1: {best_f1:.5f}')
    print('-' * 60)

    return {
        'oof_pred': oof_pred,
        'test_pred': test_pred_mean,
        'fold_scores': fold_scores,
        'oof_f1_05': oof_f1_05,
        'best_thr': best_thr,
        'best_f1': best_f1
    }


def show_feature_importance(model, feature_name, top_n=20):
    """Show top positive and negative features."""
    vectorizer = model.named_steps['tfidf']
    clf = model.named_steps['clf']
    
    feature_names = np.array(vectorizer.get_feature_names_out())
    coefs = clf.coef_[0]
    
    top_pos_idx = np.argsort(coefs)[-top_n:][::-1]
    top_neg_idx = np.argsort(coefs)[:top_n]
    
    print(f'\n=== {feature_name} - TOP FEATURES FOR CLASS 1 (depression) ===')
    for f, w in zip(feature_names[top_pos_idx], coefs[top_pos_idx]):
        print(f'  {f:30s} {w:.4f}')
    
    print(f'\n=== {feature_name} - TOP FEATURES FOR CLASS 0 (non-depression) ===')
    for f, w in zip(feature_names[top_neg_idx], coefs[top_neg_idx]):
        print(f'  {f:30s} {w:.4f}')

---

## 4. Title Model - TF-IDF on Title

In [ ]:
# Title model parameters
title_tfidf_params = {
    'lowercase': True,
    'strip_accents': 'unicode',
    'ngram_range': (1, 2),
    'min_df': 3,
    'max_df': 0.95,
    'sublinear_tf': True
}

title_clf_params = {
    'C': 2.0,
    'class_weight': 'balanced',
    'max_iter': 5000,
    'solver': 'liblinear',
    'random_state': 42
}

# Train title model
title_res = train_oof_model(
    X_train_full=train['title'],
    X_test_full=test['title'],
    y=y,
    skf=skf,
    feature_name='title',
    tfidf_params=title_tfidf_params,
    clf_params=title_clf_params
)

train['oof_title_lr'] = title_res['oof_pred']
test['test_pred_title_lr'] = title_res['test_pred']

In [ ]:
# Train on full data to get feature importance
title_model = Pipeline([
    ('tfidf', TfidfVectorizer(**title_tfidf_params)),
    ('clf', LogisticRegression(**title_clf_params))
])
title_model.fit(train['title'], y)

show_feature_importance(title_model, 'Title Model')

---

## 5. Body Model - TF-IDF on Body

In [ ]:
# Body model parameters
body_tfidf_params = {
    'lowercase': True,
    'strip_accents': 'unicode',
    'ngram_range': (1, 2),
    'min_df': 5,
    'max_df': 0.95,
    'max_features': 100_000,
    'sublinear_tf': True
}

body_clf_params = {
    'C': 3.0,
    'class_weight': 'balanced',
    'max_iter': 10000,
    'solver': 'liblinear',
    'random_state': 42
}

# Train body model
body_res = train_oof_model(
    X_train_full=train['body'],
    X_test_full=test['body'],
    y=y,
    skf=skf,
    feature_name='body',
    tfidf_params=body_tfidf_params,
    clf_params=body_clf_params
)

train['oof_body_lr'] = body_res['oof_pred']
test['test_pred_body_lr'] = body_res['test_pred']

In [ ]:
# Train on full data to get feature importance
body_model = Pipeline([
    ('tfidf', TfidfVectorizer(**body_tfidf_params)),
    ('clf', LogisticRegression(**body_clf_params))
])
body_model.fit(train['body'], y)

show_feature_importance(body_model, 'Body Model')

---

## 6. Blend Title + Body

In [ ]:
# Find best blend weights
best_blend_score = -1
best_blend_cfg = None
best_blend_oof = None

for weight_title in np.arange(0.00, 0.31, 0.01):
    weight_body = 1.0 - weight_title

    blended_oof = (
        weight_title * train['oof_title_lr'].values +
        weight_body * train['oof_body_lr'].values
    )

    for thr in np.arange(0.30, 0.81, 0.01):
        pred = (blended_oof >= thr).astype(int)
        score = f1_score(y, pred)

        if score > best_blend_score:
            best_blend_score = score
            best_blend_cfg = (round(weight_title, 2), round(weight_body, 2), round(thr, 2))
            best_blend_oof = blended_oof.copy()

print('\nBEST TITLE + BODY BLEND CONFIG:')
print(f'  title_weight = {best_blend_cfg[0]}')
print(f'  body_weight  = {best_blend_cfg[1]}')
print(f'  threshold    = {best_blend_cfg[2]}')
print(f'  blend OOF F1  = {best_blend_score:.5f}')

# Apply to test
best_w_title, best_w_body, best_thr = best_blend_cfg

blended_test_title_body = (
    best_w_title * test['test_pred_title_lr'].values +
    best_w_body * test['test_pred_body_lr'].values
)

# Save intermediate results
train[['id', 'label', 'oof_title_lr', 'oof_body_lr']].to_csv('oof_title_body_lr.csv', index=False)
test[['id', 'test_pred_title_lr', 'test_pred_body_lr']].to_csv('test_title_body_lr.csv', index=False)

test_blend_df = pd.DataFrame({
    'id': test['id'],
    'blend_pred': blended_test_title_body
})
test_blend_df.to_csv('test_blend_title_body_lr.csv', index=False)

print('\nSaved: oof_title_body_lr.csv, test_title_body_lr.csv, test_blend_title_body_lr.csv')

---

## 7. Char Model - Character-level TF-IDF

In [ ]:
# Char model parameters
char_tfidf_params = {
    'lowercase': True,
    'strip_accents': 'unicode',
    'analyzer': 'char_wb',
    'ngram_range': (3, 5),
    'min_df': 3,
    'max_df': 0.95,
    'max_features': 200_000,
    'sublinear_tf': True
}

char_clf_params = {
    'C': 3.0,
    'class_weight': 'balanced',
    'max_iter': 10000,
    'solver': 'liblinear',
    'random_state': 42
}

# Train char model
char_res = train_oof_model(
    X_train_full=train['full_text'],
    X_test_full=test['full_text'],
    y=y,
    skf=skf,
    feature_name='char_full_text',
    tfidf_params=char_tfidf_params,
    clf_params=char_clf_params
)

train['oof_char_lr'] = char_res['oof_pred']
test['test_pred_char_lr'] = char_res['test_pred']

---

## 8. Blend Title + Body + Char

In [ ]:
# Find best 3-model blend weights
best_blend_score = -1
best_blend_cfg = None
best_blend_oof = None

for w_title in np.arange(0.00, 0.21, 0.01):
    for w_body in np.arange(0.00, 0.51, 0.01):
        w_char = 1.0 - w_title - w_body

        if w_char < 0:
            continue
        if w_char < 0.40:  # char should have significant weight
            continue

        blended_oof = (
            w_title * train['oof_title_lr'].values +
            w_body * train['oof_body_lr'].values +
            w_char * train['oof_char_lr'].values
        )

        for thr in np.arange(0.30, 0.81, 0.01):
            pred = (blended_oof >= thr).astype(int)
            score = f1_score(y, pred)

            if score > best_blend_score:
                best_blend_score = score
                best_blend_cfg = (
                    round(w_title, 2),
                    round(w_body, 2),
                    round(w_char, 2),
                    round(thr, 2)
                )
                best_blend_oof = blended_oof.copy()

print('\nBEST 3-MODEL BLEND CONFIG:')
print(f'  title_weight = {best_blend_cfg[0]}')
print(f'  body_weight  = {best_blend_cfg[1]}')
print(f'  char_weight  = {best_blend_cfg[2]}')
print(f'  threshold    = {best_blend_cfg[3]}')
print(f'  blend OOF F1  = {best_blend_score:.5f}')

# Apply to test
best_w_title, best_w_body, best_w_char, best_thr = best_blend_cfg

blended_test_3model = (
    best_w_title * test['test_pred_title_lr'].values +
    best_w_body * test['test_pred_body_lr'].values +
    best_w_char * test['test_pred_char_lr'].values
)

# Save
train[['id', 'label', 'oof_title_lr', 'oof_body_lr', 'oof_char_lr']].to_csv('oof_title_body_char_lr.csv', index=False)
test[['id', 'test_pred_title_lr', 'test_pred_body_lr', 'test_pred_char_lr']].to_csv('test_title_body_char_lr.csv', index=False)

test_blend_3_df = pd.DataFrame({
    'id': test['id'],
    'blend_pred': blended_test_3model
})
test_blend_3_df.to_csv('test_blend_title_body_char_lr.csv', index=False)

print('\nSaved: oof_title_body_char_lr.csv, test_title_body_char_lr.csv, test_blend_title_body_char_lr.csv')

---

## 9. Handcrafted Features - Dense Features

In [ ]:
# Lexicons for handcrafted features
SUICIDE_PHRASES = ['kill myself', 'want die', 'want to die', 'want kill', 'end it', 'end my life']
SUICIDE_WORDS = {'suicide', 'suicidal', 'overdose', 'overdosing', 'die', 'dead', 'death'}
DEPRESSION_WORDS = {'depression', 'depressed', 'therapist', 'therapy', 'meds', 'medication',
                    'antidepressant', 'psychiatrist', 'psychologist', 'anxiety', 'ptsd', 'lexapro'}
HOPELESS_WORDS = {'nothing', 'numb', 'empty', 'emptiness', 'hopeless', 'miserable', 'exhausted',
                  'drained', 'pointless', 'worthless', 'alone', 'lonely', 'tired', 'guilt'}
DISTRESS_WORDS = {'hate', 'pain', 'cry', 'crying', 'broken', 'fucked', 'useless', 'failure',
                  'cant', 'cannot', 'don't', 'dont', 'can't'}
FIRST_PERSON_WORDS = {'i', 'me', 'my', 'myself', 'im', "i'm"}
NEGATION_WORDS = {'no', 'not', 'never', 'nothing', 'none', 'nobody', 'nowhere', 'dont', "don't", 'cant', "can't", 'cannot'}

# Helper functions
_word_re = re.compile(r'\b\w+\b', flags=re.UNICODE)

def safe_div(a, b):
    return a / b if b != 0 else 0.0

def count_words(text):
    return _word_re.findall(text.lower())

def count_substring_occurrences(text, phrases):
    text_low = text.lower()
    return sum(text_low.count(p) for p in phrases)

def count_token_hits(tokens, vocab):
    return sum(1 for t in tokens if t in vocab)

def build_features(df):
    rows = []
    for _, row in df.iterrows():
        title = row['title']
        body = row['body']
        full = row['full_text']

        title_tokens = count_words(title)
        body_tokens = count_words(body)
        full_tokens = count_words(full)

        title_nw = len(title_tokens)
        body_nw = len(body_tokens)
        full_nw = len(full_tokens)

        rows.append({
            'title_len_chars': len(title),
            'body_len_chars': len(body),
            'full_len_chars': len(full),
            'title_len_words': title_nw,
            'body_len_words': body_nw,
            'full_len_words': full_nw,
            'has_body': int(len(body.strip()) > 0),
            'title_qmarks': title.count('?'),
            'body_qmarks': body.count('?'),
            'full_qmarks': full.count('?'),
            'title_exclaims': title.count('!'),
            'body_exclaims': body.count('!'),
            'full_exclaims': full.count('!'),
            'title_ellipses': title.count('...'),
            'body_ellipses': body.count('...'),
            'full_ellipses': full.count('...'),
            'newline_count': full.count('\n'),
            'suicide_phrase_cnt': count_substring_occurrences(full, SUICIDE_PHRASES),
            'suicide_word_cnt': count_token_hits(full_tokens, SUICIDE_WORDS),
            'depression_cnt': count_token_hits(full_tokens, DEPRESSION_WORDS),
            'hopeless_cnt': count_token_hits(full_tokens, HOPELESS_WORDS),
            'distress_cnt': count_token_hits(full_tokens, DISTRESS_WORDS),
            'first_person_cnt': count_token_hits(full_tokens, FIRST_PERSON_WORDS),
            'negation_cnt': count_token_hits(full_tokens, NEGATION_WORDS),
            'suicide_phrase_ratio': safe_div(count_substring_occurrences(full, SUICIDE_PHRASES), full_nw),
            'suicide_word_ratio': safe_div(count_token_hits(full_tokens, SUICIDE_WORDS), full_nw),
            'depression_ratio': safe_div(count_token_hits(full_tokens, DEPRESSION_WORDS), full_nw),
            'hopeless_ratio': safe_div(count_token_hits(full_tokens, HOPELESS_WORDS), full_nw),
            'distress_ratio': safe_div(count_token_hits(full_tokens, DISTRESS_WORDS), full_nw),
            'first_person_ratio': safe_div(count_token_hits(full_tokens, FIRST_PERSON_WORDS), full_nw),
            'negation_ratio': safe_div(count_token_hits(full_tokens, NEGATION_WORDS), full_nw),
            'title_to_body_len_ratio': safe_div(len(title), len(body) + 1),
            'title_to_body_word_ratio': safe_div(title_nw, body_nw + 1),
        })
    return pd.DataFrame(rows)

# Build dense features
X_dense_train = build_features(train)
X_dense_test = build_features(test)

print(f'Dense train shape: {X_dense_train.shape}')
print(f'Dense test shape: {X_dense_test.shape}')

In [ ]:
# Train dense model with OOF
oof_dense = np.zeros(len(train))
test_dense_folds = np.zeros((len(test), skf.n_splits))
dense_fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dense_train, y), 1):
    X_tr = X_dense_train.iloc[tr_idx].copy()
    X_va = X_dense_train.iloc[va_idx].copy()
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_test_scaled = scaler.transform(X_dense_test)

    clf = LogisticRegression(C=1.0, class_weight='balanced', max_iter=5000, random_state=42)
    clf.fit(X_tr_scaled, y_tr)

    va_pred = clf.predict_proba(X_va_scaled)[:, 1]
    te_pred = clf.predict_proba(X_test_scaled)[:, 1]

    oof_dense[va_idx] = va_pred
    test_dense_folds[:, fold - 1] = te_pred

    fold_f1 = f1_score(y_va, (va_pred >= 0.5).astype(int))
    dense_fold_scores.append(fold_f1)

    print(f'[dense] Fold {fold}: F1 = {fold_f1:.5f}')

dense_test_pred = test_dense_folds.mean(axis=1)

dense_best_thr, dense_best_f1 = find_best_threshold(y, oof_dense)

print(f'\n[dense] OOF F1 @ 0.50: {f1_score(y, (oof_dense >= 0.5).astype(int)):.5f}')
print(f'[dense] Mean CV F1: {np.mean(dense_fold_scores):.5f}')
print(f'[dense] Best threshold: {dense_best_thr:.2f}')
print(f'[dense] Best OOF F1: {dense_best_f1:.5f}')

train['oof_dense_lr'] = oof_dense
test['test_pred_dense_lr'] = dense_test_pred

---

## 10. Meta Model - Combine All Predictions

In [ ]:
# Build meta features
meta_train = pd.DataFrame({
    'title_pred': train['oof_title_lr'],
    'body_pred': train['oof_body_lr'],
    'char_pred': train['oof_char_lr'],
    'dense_pred': train['oof_dense_lr'],
    'has_body': X_dense_train['has_body'],
    'body_len_words': X_dense_train['body_len_words'],
    'full_qmarks': X_dense_train['full_qmarks'],
    'full_exclaims': X_dense_train['full_exclaims'],
    'suicide_phrase_cnt': X_dense_train['suicide_phrase_cnt'],
    'suicide_word_cnt': X_dense_train['suicide_word_cnt'],
    'depression_cnt': X_dense_train['depression_cnt'],
    'hopeless_cnt': X_dense_train['hopeless_cnt'],
    'first_person_ratio': X_dense_train['first_person_ratio'],
    'negation_ratio': X_dense_train['negation_ratio'],
})

meta_test = pd.DataFrame({
    'title_pred': test['test_pred_title_lr'],
    'body_pred': test['test_pred_body_lr'],
    'char_pred': test['test_pred_char_lr'],
    'dense_pred': test['test_pred_dense_lr'],
    'has_body': X_dense_test['has_body'],
    'body_len_words': X_dense_test['body_len_words'],
    'full_qmarks': X_dense_test['full_qmarks'],
    'full_exclaims': X_dense_test['full_exclaims'],
    'suicide_phrase_cnt': X_dense_test['suicide_phrase_cnt'],
    'suicide_word_cnt': X_dense_test['suicide_word_cnt'],
    'depression_cnt': X_dense_test['depression_cnt'],
    'hopeless_cnt': X_dense_test['hopeless_cnt'],
    'first_person_ratio': X_dense_test['first_person_ratio'],
    'negation_ratio': X_dense_test['negation_ratio'],
})

# Train meta model with OOF
oof_meta = np.zeros(len(train))
test_meta_folds = np.zeros((len(test), skf.n_splits))
meta_fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(meta_train, y), 1):
    X_tr = meta_train.iloc[tr_idx].copy()
    X_va = meta_train.iloc[va_idx].copy()
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_test_scaled = scaler.transform(meta_test)

    meta_clf = LogisticRegression(C=1.0, class_weight='balanced', max_iter=5000, random_state=42)
    meta_clf.fit(X_tr_scaled, y_tr)

    va_pred = meta_clf.predict_proba(X_va_scaled)[:, 1]
    te_pred = meta_clf.predict_proba(X_test_scaled)[:, 1]

    oof_meta[va_idx] = va_pred
    test_meta_folds[:, fold - 1] = te_pred

    fold_f1 = f1_score(y_va, (va_pred >= 0.5).astype(int))
    meta_fold_scores.append(fold_f1)

    print(f'[meta] Fold {fold}: F1 = {fold_f1:.5f}')

meta_test_pred = test_meta_folds.mean(axis=1)

meta_best_thr, meta_best_f1 = find_best_threshold(y, oof_meta)

print(f'\n[meta] OOF F1 @ 0.50: {f1_score(y, (oof_meta >= 0.5).astype(int)):.5f}')
print(f'[meta] Mean CV F1: {np.mean(meta_fold_scores):.5f}')
print(f'[meta] Best threshold: {meta_best_thr:.2f}')
print(f'[meta] Best OOF F1: {meta_best_f1:.5f}')

train['oof_meta_lr'] = oof_meta
test['test_pred_meta_lr'] = meta_test_pred

---

## 11. Final Submission - Model Comparison

In [ ]:
# Compare all models
print('=' * 60)
print('MODEL COMPARISON')
print('=' * 60)

models_summary = []

# Title
_, title_f1 = find_best_threshold(y, train['oof_title_lr'])
models_summary.append(('Title', title_f1))

# Body
_, body_f1 = find_best_threshold(y, train['oof_body_lr'])
models_summary.append(('Body', body_f1))

# Title + Body blend
tb_blend = best_w_title * train['oof_title_lr'].values + best_w_body * train['oof_body_lr'].values
_, tb_f1 = find_best_threshold(y, tb_blend)
models_summary.append(('Title+Body Blend', tb_f1))

# Char
_, char_f1 = find_best_threshold(y, train['oof_char_lr'])
models_summary.append(('Char', char_f1))

# 3-model blend
tbc_blend = best_w_title * train['oof_title_lr'].values + best_w_body * train['oof_body_lr'].values + best_w_char * train['oof_char_lr'].values
_, tbc_f1 = find_best_threshold(y, tbc_blend)
models_summary.append(('Title+Body+Char Blend', tbc_f1))

# Dense
models_summary.append(('Dense (handcrafted)', dense_best_f1))

# Meta
models_summary.append(('Meta', meta_best_f1))

# Print summary
print(f'{'Model':<25} {'Best OOF F1':>15}')
print('-' * 40)
for name, score in sorted(models_summary, key=lambda x: -x[1]):
    print(f'{name:<25} {score:>15.5f}')

# Find best model
best_model_name, best_model_f1 = max(models_summary, key=lambda x: x[1])
print(f'\nBest model: {best_model_name} with F1 = {best_model_f1:.5f}')

In [ ]:
# Generate final submission using best model
if best_model_name == 'Meta':
    final_pred = (meta_test_pred >= meta_best_thr).astype(int)
    final_model_name = 'meta'
elif best_model_name == 'Title+Body+Char Blend':
    final_pred = (blended_test_3model >= best_thr).astype(int)
    final_model_name = 'title_body_char_blend'
elif best_model_name == 'Title+Body Blend':
    final_pred = (blended_test_title_body >= best_thr).astype(int)
    final_model_name = 'title_body_blend'
elif best_model_name == 'Char':
    _, char_thr = find_best_threshold(y, train['oof_char_lr'])
    final_pred = (dense_test_pred >= char_thr).astype(int)
    final_model_name = 'char'
elif best_model_name == 'Dense (handcrafted)':
    final_pred = (dense_test_pred >= dense_best_thr).astype(int)
    final_model_name = 'dense'
elif best_model_name == 'Body':
    _, body_thr = find_best_threshold(y, train['oof_body_lr'])
    final_pred = (test['test_pred_body_lr'].values >= body_thr).astype(int)
    final_model_name = 'body'
else:  # Title
    _, title_thr = find_best_threshold(y, train['oof_title_lr'])
    final_pred = (test['test_pred_title_lr'].values >= title_thr).astype(int)
    final_model_name = 'title'

# Save submission
submission = pd.DataFrame({
    'id': test['id'],
    'label': final_pred
})
submission.to_csv(f'submission_{final_model_name}.csv', index=False)

print(f'\nFinal submission saved as: submission_{final_model_name}.csv')
print(f'Label distribution: {submission["label"].value_counts().to_dict()}')

---

## Summary

Pipeline completed:
1. ✅ Setup - Data loading and preprocessing
2. ✅ EDA - Word frequency analysis
3. ✅ Shared Functions - Reusable train_oof_model, find_best_threshold, show_feature_importance
4. ✅ Title Model - TF-IDF + LR with feature importance
5. ✅ Body Model - TF-IDF + LR with feature importance
6. ✅ Blend Title + Body - Optimized weights
7. ✅ Char Model - Character-level TF-IDF
8. ✅ Blend Title + Body + Char - 3-way blend
9. ✅ Handcrafted Features - Dense features from lexicons
10. ✅ Meta Model - Combined all predictions
11. ✅ Final Submission - Best model selected